# 02 – Roughness Estimation

Apply the Cont-Das normalised p-variation estimator.
Reference: Pontiggia (2025) §3.4–3.5.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, '..')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from src.estimation.p_variation import estimate_roughness, k_opt, roughness_vs_K
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

## Load RV series

In [ ]:
fpath = Path('../data/processed/rv_binance_BTC_USDT_1m_2021.parquet')
if fpath.exists():
    rv = pd.read_parquet(fpath)['rv'].dropna().to_numpy()
else:
    rv = np.abs(np.random.default_rng(0).standard_normal(1000))
print(f'N={len(rv)}  K_opt={k_opt(len(rv))}')
print(f'K_opt = floor(sqrt(N)) = floor(sqrt({len(rv)})) = {k_opt(len(rv))}')
print(f'n_opt = K_opt = {k_opt(len(rv))}')

## K-stability diagnostic (paper Figure 1)

In [ ]:
K_vals = np.unique(np.round(np.logspace(np.log10(50), np.log10(min(len(rv)//2, 2500)), 60)).astype(int))
hk = roughness_vs_K(rv, K_vals)
fig, ax = plt.subplots(figsize=(7, 4))
valid = np.isfinite(hk[:, 1])
ax.plot(hk[valid, 0], hk[valid, 1], 'b-o', ms=3)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.axvline(k_opt(len(rv)), color='r', lw=1, ls=':', label=f'K_opt={k_opt(len(rv))}')
ax.set_xlabel('K'); ax.set_ylabel('H_hat'); ax.set_title('H_hat(K) diagnostic')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()

## log W(L,K,p) curve — standard and wide grids

In [ ]:
r_std = estimate_roughness(rv, p_min=0.1, p_max=4.0)
r_wide = estimate_roughness(rv, p_min=0.01, p_max=4.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, r, title in zip(axes, [r_std, r_wide], ['Standard [0.1,4]', 'Wide [0.01,4]']):
    c = r['curve']
    ax.plot(c[:,0], c[:,1], 'b-', lw=1.5)
    ax.axhline(0, color='k', lw=0.8, ls='--', label='log W=0')
    if r['H']: ax.axvline(r['H'], color='r', lw=0.8, ls=':', label=f"H={r['H']:.3f}")
    else: ax.set_title(f"{title}  [no crossing]")
    ax.set_xlabel('1/p'); ax.set_ylabel('log W(L,K,p)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
print(f"log W range (standard): [{r_std['log_W_min']:.3f}, {r_std['log_W_max']:.3f}]")

## Summary table

In [ ]:
p = Path('../results/tables/p_variation_summary.csv')
if p.exists(): display(pd.read_csv(p))
else: print('Run: python src/estimation/roughness.py')